# Jump-Diffusion Research Notebook

This notebook explores a possible Phase 2 extension of the Probabilistic Price Path Engine.

The current dashboard supports:

- GBM Monte Carlo simulation
- historical bootstrap simulation
- analytical GBM terminal benchmarking
- pathwise TP/SL probability estimation

This notebook investigates whether a jump-diffusion model can better capture occasional large intraday moves.

The goal is research only. The model should be tested here before being added to the Streamlit dashboard.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from data_loader import load_market_data
from simulator import estimate_mu_sigma, compute_log_returns, simulate_gbm_paths, simulate_bootstrap_paths
from probability_engine import pathwise_tp_sl_metrics

In [ ]:
DATA_SOURCE = "synthetic"   # "synthetic", "mt5", or "csv"

SYMBOL = "US100.cash"
TIMEFRAME = "M1"
BARS = 1000

CSV_PATH = PROJECT_ROOT / "data" / "historical" / "sample.csv"

TP_POINTS = 29.0
SL_POINTS = 29.0

HORIZON = 20
N_PATHS = 10_000
VOL_WINDOW = 90

DRIFT_MODE = "zero"
DIRECTION = "long"

SEED = 42

In [ ]:
if DATA_SOURCE == "csv":
    df = load_market_data(
        source="csv",
        csv_path=CSV_PATH,
    )
else:
    df = load_market_data(
        source=DATA_SOURCE,
        symbol=SYMBOL,
        timeframe=TIMEFRAME,
        bars=BARS,
        synthetic_start_price=21500.0,
        synthetic_seed=SEED,
        synthetic_freq="1min",
    )

# Use closed-candle logic for MT5 research.
if DATA_SOURCE == "mt5" and len(df) > VOL_WINDOW + 21:
    model_df = df.iloc[:-1].copy()
else:
    model_df = df.copy()

model_df.tail()

## Log Return Analysis

Jump-diffusion modelling starts by separating normal candle-to-candle noise from unusually large returns.

First, calculate log returns:

$$
r_t = \log\left(\frac{S_t}{S_{t-1}}\right)
$$

Then estimate recent drift and volatility using the same engine functions as the dashboard.

In [ ]:
current_price = float(model_df["Close"].iloc[-1])

mu, sigma, log_returns = estimate_mu_sigma(
    model_df["Close"],
    window=VOL_WINDOW,
    drift_mode=DRIFT_MODE,
)

recent_returns = log_returns.dropna().tail(VOL_WINDOW)

return_summary = {
    "current_price": current_price,
    "mu_per_candle": mu,
    "sigma_per_candle": sigma,
    "vol_window": VOL_WINDOW,
    "mean_recent_return": float(recent_returns.mean()),
    "std_recent_return": float(recent_returns.std(ddof=1)),
    "min_recent_return": float(recent_returns.min()),
    "max_recent_return": float(recent_returns.max()),
}

return_summary

## Jump Detection Rule

For this first research version, define a jump as a return whose absolute size is larger than a multiple of recent volatility:

$$
|r_t - \bar{r}| > k\sigma
$$

where:

- $r_t$ is the candle log return
- $\bar{r}$ is the recent mean return
- $\sigma$ is recent return volatility
- $k$ is the jump threshold multiplier

This is a simple threshold-based jump detector. It is not final model logic yet.

In [ ]:
JUMP_THRESHOLD = 2.5

recent_mean = float(recent_returns.mean())
recent_std = float(recent_returns.std(ddof=1))

jump_mask = (recent_returns - recent_mean).abs() > JUMP_THRESHOLD * recent_std
jump_returns = recent_returns[jump_mask]
normal_returns = recent_returns[~jump_mask]

jump_stats = {
    "jump_threshold_sigma": JUMP_THRESHOLD,
    "n_recent_returns": len(recent_returns),
    "n_jumps": int(jump_mask.sum()),
    "jump_intensity_per_candle": float(jump_mask.mean()),
    "normal_return_mean": float(normal_returns.mean()) if len(normal_returns) else np.nan,
    "normal_return_std": float(normal_returns.std(ddof=1)) if len(normal_returns) > 1 else np.nan,
    "jump_return_mean": float(jump_returns.mean()) if len(jump_returns) else 0.0,
    "jump_return_std": float(jump_returns.std(ddof=1)) if len(jump_returns) > 1 else 0.0,
}

jump_stats

In [ ]:
jump_table = (
    pd.DataFrame({
        "time": model_df["time"].iloc[-len(recent_returns):].values,
        "log_return": recent_returns.values,
        "is_jump": jump_mask.values,
    })
    .query("is_jump == True")
    .reset_index(drop=True)
)

jump_table